In [0]:
try:
    spark.sql("""
        CREATE EXTERNAL LOCATION bronzelayer
        URL 'abfss://bronze@storage4sadt.dfs.core.windows.net/'
        WITH (STORAGE CREDENTIAL `credential4sadt`)
    """)
except Exception as e:
    if "already exists" in str(e):
        print("External location already exists.")
    else:
        raise e


External location already exists.


In [0]:
try:
    spark.sql("""
        CREATE EXTERNAL LOCATION silverlayer
        URL 'abfss://silver@storage4sadt.dfs.core.windows.net/'
        WITH (STORAGE CREDENTIAL `credential4sadt`)
    """)
except Exception as e:
    if "already exists" in str(e):
        print("External location already exists.")
    else:
        raise e

External location already exists.


In [0]:
try:
    spark.sql("""
        CREATE EXTERNAL LOCATION goldlayer
        URL 'abfss://gold@storage4sadt.dfs.core.windows.net/'
        WITH (STORAGE CREDENTIAL `credential4sadt`)
    """)
except Exception as e:
    if "already exists" in str(e):
        print("External location already exists.")
    else:
        raise e

External location already exists.


In [0]:
%sql
SHOW EXTERNAL LOCATIONS;


name,url,comment
bronzelayer,abfss://bronze@storage4sadt.dfs.core.windows.net/,null
goldlayer,abfss://gold@storage4sadt.dfs.core.windows.net/,null
silverlayer,abfss://silver@storage4sadt.dfs.core.windows.net/,null
workspace4sadt,abfss://unity-catalog-storage@dbstorageccwb7pvcntrf4.dfs.core.windows.net/7405604715148757,null


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace4sadt.bronze
MANAGED LOCATION 'abfss://bronze@storage4sadt.dfs.core.windows.net/';


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace4sadt.silver
MANAGED LOCATION 'abfss://silver@storage4sadt.dfs.core.windows.net/';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace4sadt.gold
MANAGED LOCATION 'abfss://gold@storage4sadt.dfs.core.windows.net/';

In [0]:
%sql
-- Create schemas
CREATE SCHEMA IF NOT EXISTS workspace4sadt.bronze;
CREATE SCHEMA IF NOT EXISTS workspace4sadt.silver;
CREATE SCHEMA IF NOT EXISTS workspace4sadt.gold;

-- BRONZE: raw JSON only
CREATE TABLE IF NOT EXISTS workspace4sadt.bronze.sensors_raw (
    value STRING,              -- raw JSON
    topic STRING,
    ingest_time TIMESTAMP
)
USING DELTA;

-- SILVER: cleaned & validated
CREATE TABLE IF NOT EXISTS workspace4sadt.silver.sensors_cleaned (
    sensor_id STRING,
    sensor_timestamp TIMESTAMP,
    value DOUBLE,
    unit STRING,
    value_si DOUBLE,
    longitude DOUBLE,
    latitude DOUBLE,
    district STRING,
    topic STRING,
    ingest_time TIMESTAMP,
    is_valid BOOLEAN,
    alert_flag BOOLEAN
)
USING DELTA;

-- GOLD: aggregated metrics
CREATE TABLE IF NOT EXISTS workspace4sadt.gold.sensors_agg (
    district STRING,
    topic STRING,
    window_start TIMESTAMP,
    window_end TIMESTAMP,
    avg_value DOUBLE,
    min_value DOUBLE,
    max_value DOUBLE,
    alert_count BIGINT,
    total_count BIGINT
)
USING DELTA;
